# Prefix-Sum DP with a "best per key" Hashmap

https://atcoder.jp/contests/abc473/submissions/79043062

**Pattern:** a DP whose transition is *"jump to an earlier position carrying the same something"* — same prefix sum, same value, same remainder, same colour.

Turns O(N²) into O(N).

---

## Setup: partition problems live on the gaps

If the problem is "cut the array into contiguous blocks", stop thinking about blocks. Think about **cut positions**.

```
gap:      0     1     2     3
          ┆  1  ┆  2  ┆  0  ┆
S:        0     1     0     0
          ▲                 ▲
       always            always
        cut               cut
```

`S_i` = running total of everything to the **left** of gap `i`, mod K.

**A partition = a choice of which gaps to cut at.**

---

## The reformulation: divisibility becomes equality

Write `T_i` for the **true** running total at gap `i` (no mod), and `S_i = T_i mod K`.

**Step 1 — the block sum is a difference of running totals.**

```
sum of block between gaps l and r  =  T_r − T_l
```

**Step 2 — "difference divisible by K" is the definition of congruence.**

```
   K divides (T_r − T_l)     ⟺     T_r ≡ T_l   (mod K)
```

**Step 3 — congruent means *same remainder*.**

```
   T_r ≡ T_l  (mod K)        ⟺     T_r mod K  ==  T_l mod K
                             ⟺     S_r == S_l
```

That's the whole point of storing `S` mod K: two numbers leave the same remainder exactly when their difference is a multiple of K. A *subtraction* test becomes an *equality* test.

Check it:

```
gap:      0     1     2     3
          ┆  1  ┆  2  ┆  0  ┆
T:        0     1     3     3        true totals
S:        0     1     0     0        mod 3

   gaps 0→2:  T₂ − T₀ = 3 − 0 = 3   divisible ✓    S₀ = S₂ = 0  ✓
   gaps 0→1:  T₁ − T₀ = 1 − 0 = 1   not       ✗    S₀ = 0, S₁ = 1  ✗
```

**Why keep only the remainder?** `T_i` can reach `N·K ≈ 2×10¹⁴` and would need a big-integer in general, but only the remainder ever affects the test. Reduce as you go: `S = (S + A) % K` keeps every value under `K`.

*(Since `A_i ≥ 0` here, `S` is never negative. In problems with negative values use `((S + A) % K + K) % K`.)*

So the array disappears. You're left with a row of numbers:

> **Pick a subsequence of the S row (both ends included). Score = how many neighbours in your pick are equal.**

**Cut at gaps 0, 2, 3:**

```
gap:      0     1     2     3
cut?      ✂     no    ✂     ✂
          ┆  1     2  ┆  0  ┆
          └──────────┘└─────┘

    S values kept:   0 , 0 , 0
    pairs:        0vs0 ✓   0vs0 ✓        score 2
```

**Cut at gaps 0, 1, 3:**

```
gap:      0     1     2     3
cut?      ✂     ✂     no    ✂
          ┆  1  ┆  2     0  ┆
          └─────┘└──────────┘

    S values kept:   0 , 1 , 0
    pairs:        0vs1 ✗   1vs0 ✗        score 0
```

---

## The slow DP

`dp[i]` = best score using the first `i` elements. Try every start for the **last** block:

```
   dp[i] = max over j < i of   dp[j] + (1 if S_j == S_i else 0)
                    ▲
                    └── O(N²). This loop is the problem.
```

Run it by hand at `i = 4` on `S = 0,1,0,2,0` with `dp = 0,0,1,1,?`:

```
 j=0   S₀=0 == S₄=0  ✓   →  dp[0] + 1 = 1
 j=1   S₁=1 ≠  S₄=0  ✗   →  dp[1]     = 0
 j=2   S₂=0 == S₄=0  ✓   →  dp[2] + 1 = 2
 j=3   S₃=2 ≠  S₄=0  ✗   →  dp[3]     = 1
                                        ───
                              dp[4] =    2
```

---

## Trick 1 — Split on the condition, don't nest it

Sort those candidates into two visible piles:

```
   ┌─ PILE 1: no bonus ────────┐    ┌─ PILE 2: got the +1 ─────┐
   │  j=1   dp[1] = 0          │    │  j=0   dp[0] + 1 = 1     │
   │  j=3   dp[3] = 1          │    │  j=2   dp[2] + 1 = 2     │
   │  best = 1                 │    │  best = 2                │
   └───────────────────────────┘    └──────────────────────────┘

                   dp[4] = max(1, 2) = 2
```

Handle one pile at a time.

---

## Trick 2 — Monotone dp collapses Pile 1 to `dp[i-1]`

```
   dp:   0   0   1   1   2   2   3  ...
         └──────────────────────────┘   never decreases
```

**Why:** take any partition of the first `j` elements scoring `s`, glue one extra block on the end → a partition of the first `i` elements still scoring ≥ `s`.

So `max_{j<i} dp[j]` = **`dp[i-1]`**. One array read.

**The objection:** `dp[i-1]` is the max over *all* `j`, matching ones included — doesn't sweeping those in break it? No. A matching `j` counted *without* its bonus gives `dp[j]`, but Pile 2 already counts it as `dp[j] + 1` — strictly bigger. Extra candidates are dominated, so the final `max` is unchanged.

---

## Trick 3 — Pile 2 is always a hashmap

Pull the shared `+1` outside:

```
   Pile 2 = ( best dp[j] among earlier gaps with S_j = S_i ) + 1
            └──────────────┬───────────────────────────────┘
                    the only thing still looping
```

That question is keyed by value → group by value as you walk:

```
   ┌─────────┬────────────────────────────┬──────┐
   │ value   │ gaps carrying it           │ best │
   ├─────────┼────────────────────────────┼──────┤
   │   0     │ gap 0 (dp 0), gap 2 (dp 1) │  1   │
   │   1     │ gap 1 (dp 0)               │  0   │
   │   2     │ gap 3 (dp 1)               │  1   │
   └─────────┴────────────────────────────┴──────┘
                                             ▲
                                     only this is ever read
```

Keep just that column:

```
   f[v] = best dp among earlier gaps carrying value v
```

---

## The final recurrence

```
   dp[i]   =  max( dp[i-1] ,  f[S_i] + 1 )      ← Pile 1  vs  Pile 2
   f[S_i]  =  dp[i]                              ← record yourself
```

**Trick 4 — the update is plain assignment, not `max`.** Values enter each bucket in increasing order (dp is monotone), so whatever lands last is already the biggest.

---

## Trace

`K = 3`, `A = (1,2,2,1)` → `S = 0,1,0,2,0`. Start `dp = 0`, `f = {0:0}`.

| step | `S_i` | `f[S_i]` | `dp` = max(prev, f+1) | `f` after |
|---|---|---|---|---|
| start | — | — | 0 | `{0:0}` |
| i=1 | 1 | absent | 0 | `{0:0, 1:0}` |
| i=2 | 0 | 0 | max(0, 1) = 1 | `{0:1, 1:0}` |
| i=3 | 2 | absent | 1 | `{0:1, 1:0, 2:1}` |
| i=4 | 0 | 1 | max(1, 2) = **2** | `{0:2, ...}` |

Same `2` the hand-run loop produced — with one lookup instead of four candidates.

---

## Code

```cpp
unordered_map<long long,int> f;
f[0] = 0;                          // base state: gap 0, the empty prefix
long long S = 0;
int dp = 0;                        // dp[i-1], rolling

for (int i = 0; i < N; i++) {
    long long A; cin >> A;
    S = (S + A) % K;               // advance to the next gap

    auto it = f.find(S);
    if (it != f.end())
        dp = max(dp, it->second + 1);   // Pile 1 vs Pile 2

    f[S] = dp;                     // record — no max needed
}
```

---

## Traps

```
✗  Missing f[0] = 0
   Represents the empty prefix. Without it no block can start at element 1.
   Present-holding-0 ≠ absent — they behave differently in a +1 branch.

✗  Overflow before the mod
   K ≤ 10⁹ and A_i < K, so S + A_i reaches ~2×10⁹ — past signed 32-bit.
   Size the INTERMEDIATE, not the answer.

✗  unordered_map on AtCoder/Codeforces
   O(N) expected, O(N²) hacked. Use map, or a custom hash.

✗  Degenerate K
   K = 1 forces every A_i = 0 → every S_i = 0 → answer N. Check it.
```

---

## Recognising the pattern

- **"sum divisible by K"** → prefix sums mod K → equality of `S_l`, `S_r`.
- **"partition into contiguous blocks"** → cut positions, not blocks.
- **transition = "jump to an earlier position with the same key"** → `f[key]` hashmap.
- **"an extra block can't hurt"** → dp is monotone → the unconditional branch is `dp[i-1]`, and the cache update is assignment.

Restate the problem until the input array is no longer mentioned. That's when the DP is obvious.